# Laboratorio 0 — misma solución con OR-Tools

Asignación semanal de capacidad entre **tableros operativos** ($x_T$) y **modelos predictivos** ($x_M$).

\begin{aligned}
\max\quad &40x_T+100x_M\\
\text{s.a.}\quad &2x_T+x_M\le80 &&\text{(CPU)}\\
&2x_M\le60 &&\text{(GPU)}\\
&x_T+x_M\le50 &&\text{(analista)}\\
&x_T,x_M\ge0.
\end{aligned}

A diferencia de `scipy.optimize.linprog`, OR-Tools maximiza de forma directa.

In [1]:
import pandas as pd
from ortools.linear_solver import pywraplp

df = pd.read_csv("caso_capacidad_analitica.csv")
servicios = df.iloc[:2].copy()
capacidades = df.iloc[-1]

servicios

,servicio,valor_por_unidad,horas_cpu,horas_gpu,horas_analista
0,Tablero operativo,40.0,2,0,1
1,Modelo predictivo,100.0,1,2,1


## Modelo y resolución (GLOP)

In [2]:
solver = pywraplp.Solver.CreateSolver("GLOP")
assert solver is not None, "No se pudo crear el solver GLOP"

x_T = solver.NumVar(0, solver.infinity(), "x_T")
x_M = solver.NumVar(0, solver.infinity(), "x_M")

restricciones = {
    "CPU": solver.Add(
        2 * x_T + 1 * x_M <= int(capacidades["horas_cpu"]), "CPU"
    ),
    "GPU": solver.Add(
        0 * x_T + 2 * x_M <= int(capacidades["horas_gpu"]), "GPU"
    ),
    "analista": solver.Add(
        1 * x_T + 1 * x_M <= int(capacidades["horas_analista"]), "analista"
    ),
}

solver.Maximize(40 * x_T + 100 * x_M)
status = solver.Solve()

assert status == pywraplp.Solver.OPTIMAL, f"Estado inesperado: {status}"
print("Estado:", "OPTIMAL")
print("x_T =", x_T.solution_value())
print("x_M =", x_M.solution_value())
print("Valor óptimo =", solver.Objective().Value())

Estado: OPTIMAL
x_T = 20.000000000000007
x_M = 30.0
Valor óptimo = 3800.0


## Verificación: consumo y holguras

In [3]:
xt, xm = x_T.solution_value(), x_M.solution_value()
consumo = {
    "CPU": 2 * xt + xm,
    "GPU": 2 * xm,
    "analista": xt + xm,
}
capacidad = {
    "CPU": float(capacidades["horas_cpu"]),
    "GPU": float(capacidades["horas_gpu"]),
    "analista": float(capacidades["horas_analista"]),
}

resumen = pd.DataFrame([
    {
        "recurso": r,
        "capacidad": capacidad[r],
        "consumo": consumo[r],
        "holgura": capacidad[r] - consumo[r],
        "activa": abs(capacidad[r] - consumo[r]) < 1e-7,
        "precio_sombra": restricciones[r].dual_value(),
    }
    for r in capacidad
])

decision = pd.DataFrame([
    {"variable": "x_T", "valor": xt, "interpretación": "Tableros operativos"},
    {"variable": "x_M", "valor": xm, "interpretación": "Modelos predictivos"},
])

display(decision)
display(resumen)

,variable,valor,interpretación
0,x_T,20.0,Tableros operativos
1,x_M,30.0,Modelos predictivos


,recurso,capacidad,consumo,holgura,activa,precio_sombra
0,CPU,80.0,70.0,1.000000e+01,False,-0.0
1,GPU,60.0,60.0,0.000000e+00,True,30.0
2,analista,50.0,50.0,-7.105427e-15,True,40.0


## Recomendación

Producir **20 tableros** y **30 modelos** por semana (valor 3800). GPU y analista están activos; CPU queda con 10 h de holgura. Misma solución que con `linprog`.